# Binary-fluid Lattice Boltzmann 

This notebook uses `binary_LB.py`, a Lattice-Boltzmann binary-fluid solver based on the free-energy formulation of Swift, Orlandini, Osborn, and Yeomans (1996).


In [ ]:
from binary_LB import (
    OrlandiniBinaryFluid2D,
    exact_binodal_phi,
    exact_spinodal_phi,
    make_spinodal_example,
    make_metastable_example,
    run_and_collect,
    plot_snapshot,
    plot_velocity,
    display_animation,
    plot_phase_diagram,
    parameter_summary,
)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 120


## Deterministic spinodal decomposition

This example starts **inside the spinodal** and uses `kT = 0`, so no thermal fluctuations are needed.
For laptop speed, the first example turns hydrodynamics off. The order-parameter physics is then easiest to see.


In [ ]:
sim_spin = make_spinodal_example(
    Nx=128,
    Ny=128,
    lam=1.1,
    T=0.5,
    kappa=0.1,
    M=0.5,
    dt=0.5,
    tau=1,
    spinodal_fraction=0.5,
    phi_noise=2e-3,
    hydrodynamics=True,
    seed=4,
)
parameter_summary(sim_spin)


In [ ]:
history_spin = run_and_collect(sim_spin, steps=20000, snapshot_every=500, include_velocity=False)
fig, ax, _ = plot_snapshot(history_spin, which="phi", frame=-1)
plt.show()


In [ ]:
display_animation(history_spin, which="phi", interval=80)


## Metastable fluctuating example

This example starts **between spinodal and binodal**, so it is metastable.
Here nucleation requires fluctuations, so `kT > 0`.

A good way to judge the result is to inspect both:
- the raw `phi` field with a fixed physical color scale
- the fluctuation field `phi - <phi>`

The second view is often much better for spotting isolated nuclei.


In [ ]:
sim_meta = make_metastable_example(
    Nx=128,
    Ny=128,
    lam=1.1,
    T=0.525,
    kappa=0.2,
    M=0.2,
    dt=0.5,
    tau=0.9,
    fraction_of_binodal=0.2,
    phi_noise=2e-3,
    kT=0.004,
    hydrodynamics=False,
    seed=4,
)
parameter_summary(sim_meta)


In [ ]:
history_meta = run_and_collect(sim_meta, steps=200000, snapshot_every=2000, include_velocity=False)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_snapshot(history_meta, which="phi", frame=-1, ax=axes[0], center_on_mean=False)
plot_snapshot(history_meta, which="phi", frame=-1, ax=axes[1], center_on_mean=True, autoscale=True, title=r"$\phi-\langle\phi\rangle$")
plt.tight_layout()
plt.show()


In [ ]:
display_animation(history_meta, which="phi", interval=80, center_on_mean=False)


### Domain sizes 

In [ ]:
import numpy as np

def compute_L(phi):
    phi0 = phi - phi.mean()

    # FFT
    phi_k = np.fft.fftn(phi0)
    S_k = np.abs(phi_k)**2

    Nx, Ny = phi.shape

    kx = 2*np.pi*np.fft.fftfreq(Nx)
    ky = 2*np.pi*np.fft.fftfreq(Ny)
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    K = np.sqrt(KX**2 + KY**2)

    # flatten
    K = K.flatten()
    S_k = S_k.flatten()

    # avoid k=0
    mask = K > 0
    K = K[mask]
    S_k = S_k[mask]

    # first moment
    k_mean = np.sum(K * S_k) / np.sum(S_k)

    return 2*np.pi / k_mean


In [ ]:
L_vals = []
t_vals = history_spin["times"]

for phi in history_spin["phi"]:
    L_vals.append(compute_L(phi))

import matplotlib.pyplot as plt

plt.xscale("log")
plt.plot(t_vals, L_vals)
plt.xlabel("t")
plt.ylabel("L(t)")
plt.show()

# Largest Cluster

In [ ]:
from scipy import ndimage 

def cluster_stats(phi, threshold=0.1, phase="negative", min_size=10, connectivity=1):
    if phase == "negative":
        mask = phi < -threshold
    else:
        mask = phi > threshold

    structure = ndimage.generate_binary_structure(2, connectivity)
    labeled, ncomp = ndimage.label(mask, structure=structure)

    if ncomp == 0:
        return 0

    sizes = np.bincount(labeled.ravel())
    sizes[0] = 0
    largest = sizes.max()
    
    return largest